In [1]:
import torch
from torch import nn, optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.metrics import davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture
from collections import deque, defaultdict

# CAE

In [2]:
train = pd.read_csv('../../train_CADE_cod.csv')
val = pd.read_csv('../../val_CADE_cod.csv')
test = pd.read_csv('../../test_CADE_cod.csv')

In [3]:
classes_out = ['Infilteration','DoS attacks-SlowHTTPTest']
train = train.drop(index=train[train['Label'].isin(classes_out)].index)
val = val.drop(index=val[val['Label'].isin(classes_out)].index)
test = test.drop(index=test[test['Label'].isin(classes_out)].index)

In [4]:
train['Label'] = train['Label'].replace('FTP-BruteForce','BruteForce')
train['Label'] = train['Label'].replace('SSH-Bruteforce','BruteForce')
train['Label'] = train['Label'].replace('DoS attacks-GoldenEye','DoS')
train['Label'] = train['Label'].replace('DoS attacks-Slowloris','DoS')
# train['Label'] = train['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
train['Label'] = train['Label'].replace('DoS attacks-Hulk','DoS')
train['Label'] = train['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
train['Label'] = train['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
train['Label'] = train['Label'].replace('DDOS attack-HOIC','DDoS')
train['Label'] = train['Label'].replace('Brute Force -Web','Web')
train['Label'] = train['Label'].replace('Brute Force -XSS','Web')
train['Label'] = train['Label'].replace('SQL Injection','Web')


val['Label'] = val['Label'].replace('FTP-BruteForce','BruteForce')
val['Label'] = val['Label'].replace('SSH-Bruteforce','BruteForce')
val['Label'] = val['Label'].replace('DoS attacks-GoldenEye','DoS')
val['Label'] = val['Label'].replace('DoS attacks-Slowloris','DoS')
# val['Label'] = val['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
val['Label'] = val['Label'].replace('DoS attacks-Hulk','DoS')
val['Label'] = val['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
val['Label'] = val['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
val['Label'] = val['Label'].replace('DDOS attack-HOIC','DDoS')
val['Label'] = val['Label'].replace('Brute Force -Web','Web')
val['Label'] = val['Label'].replace('Brute Force -XSS','Web')
val['Label'] = val['Label'].replace('SQL Injection','Web')


test['Label'] = test['Label'].replace('FTP-BruteForce','BruteForce')
test['Label'] = test['Label'].replace('SSH-Bruteforce','BruteForce')
test['Label'] = test['Label'].replace('DoS attacks-GoldenEye','DoS')
test['Label'] = test['Label'].replace('DoS attacks-Slowloris','DoS')
# test['Label'] = test['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
test['Label'] = test['Label'].replace('DoS attacks-Hulk','DoS')
test['Label'] = test['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
test['Label'] = test['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
test['Label'] = test['Label'].replace('DDOS attack-HOIC','DDoS')
test['Label'] = test['Label'].replace('Brute Force -Web','Web')
test['Label'] = test['Label'].replace('Brute Force -XSS','Web')
test['Label'] = test['Label'].replace('SQL Injection','Web')

In [5]:
labels_str = [i for i in train['Label'].value_counts().index]
print(labels_str)

for i, l in enumerate(labels_str):
    train['Label'] = train['Label'].replace(l, i)
    val['Label'] = val['Label'].replace(l, i)
    test['Label'] = test['Label'].replace(l, i)

['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']


/tmp/ipykernel_336185/2307520911.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train['Label'] = train['Label'].replace(l, i)
/tmp/ipykernel_336185/2307520911.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  val['Label'] = val['Label'].replace(l, i)
/tmp/ipykernel_336185/2307520911.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('f

# CAE CADE (margin 10 e lambda 0.1)

In [6]:
def pick_pair(embeddings, labels, pick_embeddings, pick_labels):
    """
    Gera todos os pares entre embeddings e pick_embeddings,
    com labels_pair indicando se os pares são positivos (0) ou negativos (1).
    """
    N, D = embeddings.shape
    M = pick_embeddings.shape[0]

    # Expande para todas as combinações possíveis
    A = embeddings.unsqueeze(0).expand(M, N, D).reshape(M * N, D)
    B = pick_embeddings.unsqueeze(1).expand(M, N, D).reshape(M * N, D)

    labels_exp = labels.unsqueeze(0).expand(M, N)
    pick_labels_exp = pick_labels.unsqueeze(1).expand(M, N)
    labels_pair = (labels_exp != pick_labels_exp).reshape(-1).float()

    return A, B, labels_pair

def ContrastiveLoss(input1, input2, y, margin=1.0):
    # y = 0 quando input1 e input2 são da mesma classe e y = 1 caso contrário
    diff = input1 - input2
    dist_sq = torch.sum(torch.pow(diff,2),1)
    dist = torch.sqrt(dist_sq + 1e-10) # Soma 1e-10 pois se o sqrt der 0 o gradiente da NaN
    mdist = margin - dist
    mdist = F.relu(mdist)
    loss = ((1-y) * dist * dist) + (y * mdist * mdist)
    return torch.mean(loss)

def AEContrastive(input, output, embeddings, labels, pick_encoded, pick_labels, margin=1.0, lambda1=1.0):
    mseloss = nn.MSELoss()
    ae_loss = mseloss(input,output)

    A, B, labels_pair = pick_pair(embeddings, labels, pick_encoded, pick_labels)
    contrastive_loss = ContrastiveLoss(A, B, labels_pair, margin=margin)
    return ae_loss + contrastive_loss * lambda1, ae_loss, contrastive_loss

In [7]:
class contrastive_ae(nn.Module):
    def __init__(self, input_neurons, neurons):
        super().__init__()
        self.encoder0 = nn.Linear(input_neurons,64)
        self.encoder1 = nn.Linear(64,32)
        self.encoder2 = nn.Linear(32,16)
        self.encoder3 = nn.Linear(16,neurons)

        self.decoder0 = nn.Linear(neurons,16)
        self.decoder1 = nn.Linear(16,32)
        self.decoder2 = nn.Linear(32,64)
        self.decoder3 = nn.Linear(64,input_neurons)

        self.activation0 = nn.ReLU()
    
    def forward(self, X):
        X = self.activation0(self.encoder0(X))
        X = self.activation0(self.encoder1(X))
        X = self.activation0(self.encoder2(X))
        X = self.encoder3(X)
        encoded = X
        X = self.activation0(self.decoder0(X))
        X = self.activation0(self.decoder1(X))
        X = self.activation0(self.decoder2(X))
        X = self.decoder3(X)


        return X, encoded

In [8]:
class encoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.encoder0 = list(model.children())[0]
        self.encoder1 = list(model.children())[1]
        self.encoder2 = list(model.children())[2]
        self.encoder3 = list(model.children())[3]
        self.activation0 = list(model.children())[8]
    
    def forward(self, X):
        X = self.activation0(self.encoder0(X))
        X = self.activation0(self.encoder1(X))
        X = self.activation0(self.encoder2(X))
        X = self.encoder3(X)
        return X

In [9]:
from torch.utils.data import Dataset, DataLoader,TensorDataset
import random
def build_class_reference_batches(X_train, y_train, batch_size=512):
    classes = torch.unique(y_train).tolist()
    n_classes = len(classes)
    total_samples = len(X_train)
    dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    

    print(classes)

    # Mapeia cada classe para seus índices
    class_to_indices = {
        int(cls.item()): (y_train == cls).nonzero(as_tuple=True)[0].tolist()
        for cls in torch.unique(y_train)
    }

    reference_list = []
    total_batches = len(train_loader)

    for batch_idx in range(total_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, total_samples)
        batch_indices = set(range(start, end))

        class_samples = []
        for cls in classes:
            candidates = [i for i in class_to_indices[cls] if i not in batch_indices]
            if not candidates:
                raise ValueError(f"Classe {cls} não possui amostras fora do batch {batch_idx}")
            chosen = candidates[torch.randint(len(candidates), (1,)).item()]
            class_samples.append(chosen)

        reference_list.append(class_samples)


    return train_loader, reference_list

In [10]:
def normalize(t):
    t_norm = (t - t.mean()) / t.std()
    return t_norm


In [11]:
array_hidden_classes = [[0,2],[0,3],[0,4],[0,5],[2,3],[2,4],[2,5],[3,4],[3,5],[4,5]]
filenames = ['0_2','0_3','0_4','0_5','2_3','2_4','2_5','3_4','3_5','4_5']
# array_hidden_classes = [[0]]
# filenames = ['0']
total_classes = len(labels_str)
for i in range(len(array_hidden_classes)):
    filename = f'{filenames[i]}_hidden'
    print(filename)
    hidden_classes = array_hidden_classes[i] #Classes ocultas do treinamento

    hidden_classes_index = list(train[train['Label'].isin(hidden_classes)].index)

    train_hidden = train.loc[hidden_classes_index]

    train_not_hidden = train.drop(hidden_classes_index)

    X_train = train_not_hidden.drop(columns=['Label'])
    y_train = train_not_hidden['Label']
    X_val = val.drop(columns=['Label'])
    y_val = val['Label'].values
    X_test = test.drop(columns=['Label'])
    y_test = test['Label'].values

    torch.manual_seed(123)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(device)
    X_train = torch.tensor(np.array(X_train), dtype=torch.float)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float)

    train_loader, pick_list = build_class_reference_batches(X_train,y_train, batch_size=512)
    pick_samples = []
    pick_labels = []
    for l in pick_list:
        pick_samples.append(X_train[l])
        pick_labels.append(y_train[l])

    pick_samples = torch.stack(pick_samples)    
    pick_labels = torch.stack(pick_labels)

    print('pick completo')

    neurons = total_classes - len(hidden_classes)
    print(f'{X_train.shape[1]} {neurons}')
    model = contrastive_ae(input_neurons=X_train.shape[1],neurons=neurons)
    model.to(device)

    learning_rate = 0.0001
    opt = optim.Adam(model.parameters(),lr=learning_rate)
    margin = 10 # Parametro de afastamento das classes
    cae_lambda_1 = 0.1 # Parametro de influencia na distancia das classes
    
    
    for epoch in range(250):
        running_loss = 0
        running_ael = 0
        running_cl = 0
        contador = 0
        for data in train_loader:
            inputs, labels = data
            inputs = inputs.to(device)
            labels = labels.to(device)
            ps = pick_samples[contador]
            pl = pick_labels[contador]
            ps = ps.to(device)
            pl = pl.to(device)
            contador += 1
            #print(f'{contador}/{len(train_loader)}')
            opt.zero_grad()
            outputs, encoded = model(inputs)
            _, pick_encoded = model(ps)
            tam_pe = pick_encoded.shape[0]
            enc_total = torch.cat((encoded,pick_encoded), dim=0)
            enc_total_norm = normalize(enc_total)
            encoded_norm = enc_total_norm[:-tam_pe]
            pick_encoded_norm = enc_total_norm[-tam_pe:]
            loss, ael, cl = AEContrastive(input=inputs, output=outputs, embeddings = encoded_norm, labels=labels, pick_encoded = pick_encoded_norm, pick_labels = pl, margin=margin, lambda1=cae_lambda_1)
            #loss = dummyMSELoss(inputs,outputs)
            loss.backward()
            opt.step()
            running_loss += loss.item()
            running_ael += ael.item()
            running_cl += cl.item()
            #print(f'loss = {loss.item()} batch_size = {size}')
        print(f'filename: {filename} Epoch {epoch+1} loss:{running_loss/len(train_loader)} ael:{running_ael/len(train_loader)} cl:{running_cl/len(train_loader)}')

    model_e = encoder(model=model)
    model_e.to(device)

    model_e.eval()

    X_train = torch.tensor(np.array(X_train), dtype=torch.float)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float)

    train_encoded = model_e(X_train.to(device))

    train_encoded = train_encoded.detach().cpu().numpy()

    train_encoded_df = pd.DataFrame(train_encoded)
    train_encoded_df['Label'] = y_train.detach().cpu().numpy().astype(int)
    display(train_encoded_df)

    score = davies_bouldin_score(train_encoded_df.drop(columns=['Label']), train_encoded_df['Label'])
    print("Davies-Bouldin Score:", score)

    X_val = torch.tensor(np.array(X_val), dtype=torch.float)

    val_encoded = model_e(X_val.to(device))

    val_encoded = val_encoded.detach().cpu().numpy()

    val_encoded_df = pd.DataFrame(val_encoded)

    val_encoded_df['Label'] = y_val

    display(val_encoded_df)

    X_test = torch.tensor(np.array(X_test), dtype=torch.float)

    test_encoded = model_e(X_test.to(device))

    test_encoded = test_encoded.detach().cpu().numpy()

    test_encoded_df = pd.DataFrame(test_encoded)

    test_encoded_df['Label'] = y_test

    display(test_encoded_df)

    train_encoded_df.to_csv(f'train_encoded_CADE_cod_joined_attacks_pick_class_squared_normalized_2_hidden_{filename}.csv',index=False)
    val_encoded_df.to_csv(f'val_encoded_CADE_cod_joined_attacks_pick_class_squared_normalized_2_hidden_{filename}.csv',index=False)
    test_encoded_df.to_csv(f'test_encoded_CADE_cod_joined_attacks_pick_class_squared_normalized_2_hidden_{filename}.csv',index=False)

0_2_hidden


cuda
[1.0, 3.0, 4.0, 5.0]


pick completo
82 4


filename: 0_2_hidden Epoch 1 loss:3.650869241097699 ael:0.034645244148154945 cl:36.16223941263945


filename: 0_2_hidden Epoch 2 loss:2.9987848297409387 ael:0.020247758696179674 cl:29.785370207869487


...

/tmp/ipykernel_336185/680421385.py:91: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  X_train = torch.tensor(np.array(X_train), dtype=torch.float)
/tmp/ipykernel_336185/680421385.py:92: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  y_train = torch.tensor(np.array(y_train), dtype=torch.float)


,0,1,2,3,Label
0,0.107071,0.163419,0.181711,0.196098,1
1,0.218040,0.244552,0.187217,0.153868,0
2,0.223544,0.226174,0.294698,0.290629,2
3,0.108844,0.170357,0.179945,0.193039,1
4,0.223545,0.226176,0.294698,0.290629,2
...,...,...,...,...,...
1896092,0.217946,0.244494,0.187148,0.153823,0
1896093,0.223544,0.226174,0.294698,0.290629,2
1896094,0.223544,0.226173,0.294698,0.290629,2
1896095,0.218091,0.244584,0.187254,0.153893,0


Davies-Bouldin Score: 0.0728871137665506


,0,1,2,3,Label
0,0.218095,0.244053,0.186225,0.156033,0
1,0.223543,0.226174,0.294697,0.290628,2
2,0.107555,0.165033,0.182693,0.203914,1
3,0.292210,0.080872,0.144376,0.216097,3
4,0.107639,0.164739,0.182299,0.200583,1
...,...,...,...,...,...
519951,0.102363,0.165837,0.180993,0.200152,1
519952,0.091516,0.161189,0.175495,0.198869,4
519953,0.217970,0.244509,0.187165,0.153834,0
519954,0.115294,0.167738,0.185312,0.198889,1


,0,1,2,3,Label
0,0.218030,0.244546,0.187210,0.153863,0
1,0.218255,0.244139,0.186207,0.155917,0
2,0.124428,0.175457,0.185959,0.200869,1
3,0.118782,0.176710,0.182547,0.195646,1
4,0.215659,0.242332,0.186687,0.159383,0
...,...,...,...,...,...
649942,0.110653,0.166453,0.182877,0.200362,1
649943,0.216380,0.243031,0.187258,0.159671,0
649944,0.215920,0.242645,0.186911,0.159478,0
649945,0.299886,0.080099,0.142775,0.214863,3
